In [7]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType, DoubleType

In [8]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration for the EDSTD project") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql(
    """
    SHOW TABLES FROM projeto
    """
).show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|  projeto|   amazon_books|      false|
|  projeto| amazon_credits|      false|
|  projeto|  amazon_titles|      false|
|  projeto|netflix_credits|      false|
|  projeto| netflix_titles|      false|
+---------+---------------+-----------+



In [4]:
# spark.sql(
#    """
#    DROP TABLE IF EXISTS projeto.book_movie_adapt
#    """
#)

DataFrame[]

In [67]:
spark.sql("""
    CREATE EXTERNAL TABLE projeto.book_movie_adapt (
        authorLabel VARCHAR(75),
        bookLabel VARCHAR(190),
        book_year VARCHAR(5),
        filmLabel VARCHAR(80),
        release_year VARCHAR(5)
    )
    USING DELTA
    PARTITIONED BY (release_year)
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/projeto.db/book_movie_adapt/'
"""
)

DataFrame[]

In [68]:
spark.sql(
    """
    DESCRIBE FORMATTED projeto.book_movie_adapt
    """
).toPandas()

,col_name,data_type,comment
0,authorLabel,string,None
1,bookLabel,string,None
2,book_year,string,None
3,filmLabel,string,None
4,release_year,string,None
5,# Partition Information,,
6,# col_name,data_type,comment
7,release_year,string,None
8,,,
9,# Detailed Table Information,,


In [71]:
spark.sql(
    """
    SELECT * FROM projeto.book_movie_adapt
    """
).show()

+----------------+--------------------+---------+--------------------+------------+
|     authorLabel|           bookLabel|book_year|           filmLabel|release_year|
+----------------+--------------------+---------+--------------------+------------+
|  T. E. Lawrence|Seven Pillars of ...|     1922|  Lawrence of Arabia|        1962|
|  T. E. Lawrence|Seven Pillars of ...|     1926|  Lawrence of Arabia|        1962|
|           Homer|               Iliad|     null|The Fury of Achilles|        1962|
|     Ian Fleming|              Dr. No|     1958|              Dr. No|        1962|
|      Cao Xueqin|Dream of the Red ...|     1792|The Dream of the ...|        1962|
|      Cao Xueqin|Dream of the Red ...|     1792|Dream of the Red ...|        1962|
|      Cao Xueqin|Dream of the Red ...|     1800|The Dream of the ...|        1962|
|      Cao Xueqin|Dream of the Red ...|     1800|Dream of the Red ...|        1962|
|      Cao Xueqin|Dream of the Red ...|     1868|The Dream of the ...|      

In [72]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.book_movie_adapt
    """
).show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-10-31 22:04:...|  null|    null|       WRITE|{mode -> Overwrit...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 106,...|        null|Apache-Spark/3.4....|
|      0|2025-10-31 22:03:...|  null|    null|CREATE TABLE|{isManaged -> fal...|null|    null|     null|       null|  Serializable|         true|                  {}|        null|Apache-Spark/3.4.